In [ ]:
# Cell 0
import torch, gc
gc.collect()
torch.cuda.empty_cache()

In [7]:
# Cell 1 - Cloning Repo

repo_url = "https://github.com/siddheshkadane01/Site-Reliability-Server/"
repo_dir = repo_url.rstrip("/").split("/")[-1]

import shutil, os

if os.path.exists(repo_dir):
    shutil.rmtree(repo_dir)

!git clone -b main {repo_url}

%cd {repo_dir}

Cloning into 'Site-Reliability-Server'...
remote: Enumerating objects: 919, done.
remote: Counting objects: 100% (919/919), done.
remote: Compressing objects: 100% (228/228), done.[K
remote: Total 919 (delta 729), reused 865 (delta 678), pack-reused 0 (from 0)
Receiving objects: 100% (919/919), 8.30 MiB | 554.00 KiB/s, done.
Resolving deltas: 100% (729/729), done.
Filtering content: 100% (2/2), 3.40 KiB | 1024 bytes/s, done.
/Users/sriram_kommalapudi/Projects/Site-Reliability-Server/Site-Reliability-Server/Site-Reliability-Server


In [ ]:
# Cell 1.5
!grep -n "num_generations must be" train_grpo.py

In [ ]:
# Cell 2 - All installs in one cell, correct order:
%pip install -r requirements.txt
%pip install -r requirements_rl.txt

# Pin TRL (important for GRPO stability)
%pip install trl==0.18.2 --quiet

!pip show trl | grep Version

Note: you may need to restart the kernel to use updated packages.
  Attempting uninstall: trl
    Found existing installation: trl 0.23.0
    Uninstalling trl-0.23.0:
      Successfully uninstalled trl-0.23.0
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Version: 0.18.2


In [ ]:
# Cell 3 - Start FastAPI server with auto-restart monitor
import subprocess, time, requests, threading, os

def start_server():
    log_file = open("server.log", "a")
    process = subprocess.Popen(
        ["uvicorn", "main:app", "--host", "0.0.0.0", "--port", "7860",
         "--workers", "1", "--timeout-keep-alive", "300"],
        stdout=log_file, stderr=log_file
    )
    return process

def monitor_and_restart_server():
    global server_process
    while True:
        try:
            res = requests.get("http://localhost:7860/health", timeout=5)
            if res.status_code != 200:
                raise Exception("Unhealthy")
        except:
            print("⚠️ Server down — restarting...")
            try:
                server_process.kill()
            except:
                pass
            time.sleep(3)
            server_process = start_server()
            time.sleep(10)
            print("✅ Server restarted")
        time.sleep(15)  # check every 15 seconds

# Kill any existing server on port 7860
os.system("fuser -k 7860/tcp 2>/dev/null || true")
time.sleep(2)

# Start server
server_process = start_server()
time.sleep(12)

# Health check
try:
    res = requests.get("http://localhost:7860/health", timeout=5)
    print("✅ Server running" if res.status_code == 200 
          else f"❌ Unexpected status: {res.status_code}")
except Exception as e:
    print("❌ Server failed:", e)
    print(open("server.log").read())

# Start monitor thread (daemon=True means it dies when Colab session ends)
monitor_thread = threading.Thread(target=monitor_and_restart_server, daemon=True)
monitor_thread.start()
print("✅ Server monitor running — will auto-restart if server crashes")
print(open("server.log").read()[:300])

fuser: [-cfu] file ...
	-c	file is treated as mount point
	-f	the report is only for the named files
	-u	print username of pid in parenthesis
✅ Server running
✅ Server monitor running — will auto-restart if server crashes
INFO:     Started server process [86634]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:7860 (Press CTRL+C to quit)
INFO:     127.0.0.1:56662 - "GET /health HTTP/1.1" 200 OK



In [ ]:
# Diagonsis Cell
import requests

session = requests.Session()

# Test reset
r = session.post("http://localhost:7860/reset", 
                 json={"task_id": "enterprise"}, timeout=10)
print("Reset status:", r.status_code)
obs = r.json()
print("Obs keys:", list(obs.keys()) if isinstance(obs, dict) else obs)

# Test a step
r2 = session.post("http://localhost:7860/step",
                  json={"action_type": "CHECK_LOGS", 
                        "target_service": "api-gateway"}, timeout=10)
print("Step status:", r2.status_code)
result = r2.json()
print("Reward:", result.get("reward"))
print("Done:", result.get("done"))

Reset status: 200
Obs keys: ['step', 'max_steps', 'task_id', 'metrics', 'logs', 'deploy_history', 'current_config', 'service_graph', 'active_alerts', 'health_summary', 'incident_context', 'apps_state', 'protocol_status', 'mode', 'collaboration_state']
Step status: 200
Reward: {'step_reward': -0.2, 'cumulative': -0.2, 'breakdown': {'health_delta': 0.0, 'task_progress': 0.0, 'cost_efficiency': 0.0, 'latency_delta': 0.0, 'invalid_penalty': -0.0, 'risk_penalty': -0.0, 'protocol_penalty': -0.2, 'protocol_progress_bonus': 0.0, 'completion_bonus': 0.0, 'coordination_bonus': 0.0, 'coordination_penalty': -0.0}}
Done: False


In [ ]:
# Cell 4 - Wandb login using secret:
import wandb, os
from google.colab import userdata

os.environ["WANDB_API_KEY"] = userdata.get("WANDB_API_KEY")
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")

wandb.login()

from huggingface_hub import login
login(token=os.environ["HF_TOKEN"])

print("✅ HuggingFace login successful")

ModuleNotFoundError: No module named 'google.colab'

In [ ]:
# Cell 5 - Final Training Run

import os
os.environ["WANDB_PROJECT"] = "openenv-enterprise-grpo"
os.environ["WANDB_RUN_NAME"] = "grpo-qwen3b-T4-gen2"
os.environ["WANDB_MODE"] = "online"

!python train_grpo.py \
    --epochs 3 \
    --num_generations 2 \
    --per_device_train_batch_size 2 \
    --dataset_size 32 \
    --temperature 1.0 \
    --top_p 0.92 \
    --lora_dropout 0.05 \
    --learning_rate 5e-6 \
    --env_url http://localhost:7860 \
    --model_name Qwen/Qwen2.5-Coder-3B-Instruct \
    --max_new_tokens 32 \
    --max_prompt_length 512 \
    --output_dir ./artifacts/grpo-qwen3b-T4-final